# Fig 4A / 4B — Meta-analysis of similar populations, ADSP + UKB (data processing)

Simulated meta-analysis mixing ADSP and UKB samples at known proportions. SuSiE-rRSS models the study LD as a weighted mixture of the reference sketches.

**Outputs:**
- `figure_data/Fig4A.ADSP_UKB_Weight_Estimation.rds` — estimated vs. true ADSP proportion (+ `_accuracy.csv`)
- `figure_data/Fig4B.ADSP_UKB_Mix_Power_FDR.rds` — power / FDR of mixed vs. single LD (bootstrap 95% CI)

In [1]:
source("../../R/utils.R")

res_file   <- "/hpc/users/sunh17/haochen/rss_qc/2026_output/ADSP_UKB_fine/summary/result_susie_mix_summary_summary.rds"
n_GWAS     <- 2000

## Fig 4A: weight estimation

In [2]:
res <- readRDS(res_file) %>%
  mutate(weight = as.numeric(str_extract(input_file, "(?<=prop_)[0-9.]+")))

## omega recovery only defined for the automatic multi-ref fit (mix_refpanel)
om <- res %>%
  filter(method == "mix_refpanel", !is.na(omega_1)) %>%
  transmute(h2g, causal, weight = as.numeric(weight), omega_1)

## accuracy table for the caption (mean / sd / MAE of omega vs truth)
om_acc <- om %>%
  group_by(h2g, causal, weight) %>%
  summarize(mean_omega = mean(omega_1), sd_omega = sd(omega_1),
            MAE = mean(abs(omega_1 - weight)), n = n(), .groups = "drop")
write_csv(om_acc, file.path(data_dir, "Fig4A.ADSP_UKB_Weight_Estimation_accuracy.csv"))

om <- om %>%
  mutate(
    weight  = as.numeric(weight),
    h2g_num = parse_h2g(h2g),
    h2g = fct_reorder(paste0("Per-SNP heritability = ", formatC(h2g_num, format = "f", digits = 2)), h2g_num)
  )
save_data(om, "Fig4A.ADSP_UKB_Weight_Estimation")

## Fig 4B: power / FDR

In [3]:
method_labels <- c(
  gen_refpanel    = "Oracle",
  UKB_refpanel    = "UKB only",
  ADSP_refpanel   = "ADSP only",
  manual_refpanel = "SuSiE-rRSS"
)
method_order <- names(pal_meta_adsp_ukb)      # SuSiE-rRSS, Oracle, ADSP only, UKB only

cell_reps <- readRDS(res_file) %>%
  mutate(weight = as.numeric(str_extract(input_file, "(?<=prop_)[0-9.]+"))) %>%
  { if ("fit_ok" %in% names(.)) filter(., fit_ok) else . } %>%
  mutate(method = recode(method, !!!method_labels)) %>%
  filter(!round(weight, 3) %in% c(0.10, 0.90),          # drop ghost endpoints
         method %in% method_order)                       # drops mix_refpanel, gen_insample, etc.

## bootstrap 95% CIs (boot_cell_mix() in R/utils.R)
n_boot <- 1000
set.seed(1)
summary_ci <- cell_reps %>%
  group_by(method, weight, h2g, causal) %>%
  group_modify(~ boot_cell_mix(.x, B = n_boot)) %>%
  ungroup()

In [4]:
## ---- plot-ready: long Power / FDR, NCP and causal facets ---------------------
plot_df <- summary_ci %>%
  mutate(
    ncp_num = n_GWAS * parse_h2g(h2g),
    ncp = fct_reorder(paste0("NCP = ", formatC(ncp_num, format = "f", digits = 0)), ncp_num)
  )

w_levels <- sort(unique(plot_df$weight))
pl <- plot_df %>%
  transmute(method, weight, ncp, causal,
            Power_est = power, Power_lo = power_lo, Power_hi = power_hi,
            FDR_est   = FDR,   FDR_lo   = FDR_lo,   FDR_hi   = FDR_hi) %>%
  pivot_longer(-c(method, weight, ncp, causal),
               names_to = c("metric", ".value"), names_sep = "_") %>%
  mutate(
    metric = factor(metric, levels = c("Power", "FDR")),
    causal = factor(paste0(causal, " causal"),
                    levels = paste0(sort(unique(as.integer(causal))), " causal")),
    weight = factor(weight, levels = w_levels, labels = paste0(round(w_levels * 100), "%")),
    method = factor(method, levels = method_order)
  )
stopifnot(!any(is.na(pl$method)))

save_data(pl, "Fig4B.ADSP_UKB_Mix_Power_FDR")